# VK RQKmeans quantization

Аналог `notebooks/RQKmeansPipeline.ipynb`. На входе — `tuned_content_embeddings.pkl`, на выходе — `tuned_index_rqkmeans.json` с семантическими ID длины 4 (3 кодбука + collision_solver).

In [ ]:
%pip install scikit-learn==1.7.2

In [ ]:
from collections import defaultdict

import numpy as np
import json
import pickle

from sklearn.cluster import KMeans

In [ ]:
embeddings_input_path = '../data/VK/tuned_content_embeddings.pkl'
semantic_index_output_path = '../data/VK/tuned_index_rqkmeans.json'

In [ ]:
with open(embeddings_input_path, 'rb') as f:
    data = pickle.load(f)

item_ids = np.array(data['item_id'], dtype=np.int64)
X = np.array(data['embedding'], dtype=np.float32)
X.shape

## RQKMeans

In [ ]:
class RQKMeans:
    def __init__(
            self,
            num_clusters,
            num_codebooks,
            init='k-means++',
            max_iter=300,
            tol=1e-4,
            verbose=0,
            random_state=42
    ):
        self.models = [
            KMeans(
                n_clusters=num_clusters,
                init=init,
                max_iter=max_iter,
                tol=tol,
                verbose=verbose,
                random_state=random_state + i,
            ) for i in range(num_codebooks)
        ]

    def fit(self, X, y=None):
        for model in self.models:
            y = model.fit_predict(X)
            X = X - model.cluster_centers_[y]
        return self

    def predict(self, X):
        result = []
        centroids = []
        for model in self.models:
            result.append(model.predict(X))
            centroids.append(model.cluster_centers_[result[-1]])
            X = X - centroids[-1]
        return np.stack(result, axis=-1)

In [ ]:
rq_kmeans = RQKMeans(num_clusters=256, num_codebooks=3, max_iter=1000)
rq_kmeans.fit(X)

In [ ]:
clusters = rq_kmeans.predict(X)
clusters.shape

In [ ]:
clusters[:10]

## Сохранение semantic IDs (с collision_solver)

Каждый айтем получает 4 кода: 3 от RQKMeans + 1 collision_solver, разрешающий коллизии внутри одного 3-tuple.

In [ ]:
inter = {}
sem_2_ids = defaultdict(list)
for idx, cls in zip(item_ids, clusters):
    inter[int(idx)] = cls.tolist()
    sem_2_ids[tuple(cls.tolist())].append(int(idx))

for semantics, ids in sem_2_ids.items():
    assert len(ids) <= 256
    collision_solvers = np.random.permutation(256)[:len(ids)].tolist()
    for item_id, collision_solver in zip(ids, collision_solvers):
        inter[item_id].append(collision_solver)

with open(semantic_index_output_path, 'w') as f:
    json.dump(inter, f)

print(f'tuned_index_rqkmeans.json сохранён: {semantic_index_output_path}')
print(f'  айтемов: {len(inter)}, длин: {set(len(v) for v in inter.values())}')